# RFP vs Proposal Coverage Mapper (all-in-one Kaggle notebook)

**Before running (one time):**
1. **Settings > Internet**: On
2. **Settings > Accelerator**: None (no GPU needed)
3. **Add-ons > Secrets**: `GEMINI_API_KEY` **ticked** for this notebook

Then click **Run All**. No dataset upload is needed: this notebook creates every project file itself.

The last cell zips everything. Download it, then click **Save Version**.

## 1. Create the project folder

In [ ]:
import os, sys

# Create the project folders and work inside them
PROJECT = '/kaggle/working/rfp-coverage-mapper'
for folder in ['src', 'synthetic']:
    os.makedirs(os.path.join(PROJECT, folder), exist_ok=True)
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
print('Working in:', os.getcwd())

## 2. Write the project files
Each cell below saves one file. You'll see `Writing ...` or `Overwriting ...`.

In [ ]:
%%writefile requirements.txt
google-genai        # official Gemini SDK
python-dotenv       # loads the API key from .env
python-docx         # read/write Word files
python-pptx         # read/write PowerPoint files
pdfplumber          # read PDF files (Day 2)
reportlab           # write PDF files (synthetic RFP)
pandas              # tables (Day 5)
openpyxl            # Excel output (Day 5)
# sentence-transformers gets added on Day 4 (it is a large download)


In [ ]:
%%writefile .gitignore
.env
venv/
__pycache__/
*.pyc
.DS_Store
~$*


In [ ]:
%%writefile .env.example
# Copy this file to ".env" and paste your real key below.
# The .env file is ignored by Git, so your key never ends up on GitHub.
GEMINI_API_KEY=paste-your-key-here

# Which Gemini model to use. Run "python check_setup.py" to see which
# models your key can access, then change this line if needed.
GEMINI_MODEL=gemini-3.8-flash


In [ ]:
%%writefile src/__init__.py
# Marks this folder as a Python package (needed for imports)


In [ ]:
%%writefile synthetic/__init__.py
# Marks this folder as a Python package (needed for imports)


In [ ]:
%%writefile src/llm.py
"""
llm.py - the ONE place in the project that talks to Gemini.

Every other file calls ask_json(). Keeping all AI calls here means:
  - the API key is loaded in one place (never written in code)
  - busy/rate-limit errors are handled once, with an automatic backup model
  - every answer is cached to disk, so reruns are free and repeatable
  - switching model/provider later means changing only this file
"""
import hashlib
import json
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()  # reads a .env file if one exists (local use); harmless if not

MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
# Backup model(s), used only if the main one keeps failing (busy / quota)
FALLBACK_MODELS = [m.strip() for m in os.getenv("GEMINI_FALLBACK_MODELS", "gemini-3.5-flash-lite").split(",") if m.strip()]
CACHE_DIR = Path(os.getenv("LLM_CACHE_DIR", "cache"))
USE_CACHE = os.getenv("LLM_CACHE", "on").lower() != "off"
ATTEMPTS_PER_MODEL = 3

_client = None
_key_source = None


def _find_api_key():
    """Return (key, where_it_came_from). Checks .env/environment, then Kaggle Secrets."""
    key = os.getenv("GEMINI_API_KEY")
    if key and key != "paste-your-key-here":
        return key, ".env file / environment variable"
    try:
        from kaggle_secrets import UserSecretsClient  # only exists on Kaggle
        key = UserSecretsClient().get_secret("GEMINI_API_KEY")
        if key:
            return key, "Kaggle Secrets"
    except Exception:
        pass
    return None, None


def get_client():
    """Create the Gemini client once and reuse it."""
    global _client, _key_source
    if _client is None:
        key, _key_source = _find_api_key()
        if not key:
            raise RuntimeError(
                "No GEMINI_API_KEY found. Locally: add it to .env. "
                "On Kaggle: Add-ons > Secrets, add GEMINI_API_KEY and tick it for this notebook."
            )
        _client = genai.Client(api_key=key)
    return _client


def key_source():
    """Where the API key was loaded from (call after get_client())."""
    return _key_source


def _cache_file(prompt, system, temperature, tag):
    """
    A cache 'fingerprint': the same main model + prompt + settings always gives
    the same file name. If anything changes (even one word), it's a new entry.
    """
    raw = json.dumps({"model": MODEL, "system": system, "prompt": prompt,
                      "temperature": temperature, "tag": tag}, sort_keys=True)
    return CACHE_DIR / (hashlib.sha256(raw.encode("utf-8")).hexdigest()[:16] + ".json")


def _is_temporary(msg):
    """Errors worth waiting for: rate limits (429) and busy servers (503/504)."""
    signs = ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "overloaded", "504", "DEADLINE_EXCEEDED")
    return any(s in msg for s in signs)


def ask_json(prompt, system=None, temperature=0.2, cache_tag="", use_cache=True):
    """
    Send a prompt to Gemini and get back a Python dict/list.

    1. If we've asked this exact question before, return the saved answer (no API call).
    2. Otherwise call the main model in JSON mode, so the reply is machine-readable.
    3. Temporary error (busy / rate limit)? Wait and retry, doubling the wait
       each time ("exponential backoff"), and print the REAL reason.
    4. Still failing after 3 tries? Switch to the backup model and repeat.
    5. Save the answer to the cache for next time (noting which model answered).

    cache_tag lets us deliberately ask the same prompt again for a fresh answer
    (e.g. "deck-attempt-2" when regenerating a deck that broke the rules).
    """
    caching = USE_CACHE and use_cache
    path = _cache_file(prompt, system, temperature, cache_tag)
    if caching and path.exists():
        print(f"  [llm] cache hit ({path.name}) - no API call used")
        return json.loads(path.read_text(encoding="utf-8"))["response"]

    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        response_mime_type="application/json",
    )
    for model in [MODEL] + FALLBACK_MODELS:
        wait = 10
        for attempt in range(1, ATTEMPTS_PER_MODEL + 1):
            try:
                response = get_client().models.generate_content(model=model, contents=prompt, config=config)
                if not response.text:
                    raise ValueError("empty reply")
                result = json.loads(response.text)
                if model != MODEL:
                    print(f"  [llm] answered by backup model {model}")
                if caching:
                    CACHE_DIR.mkdir(parents=True, exist_ok=True)
                    path.write_text(json.dumps({"model_used": model, "tag": cache_tag,
                                                "prompt_preview": prompt[:300],
                                                "response": result}, indent=2), encoding="utf-8")
                return result
            except (json.JSONDecodeError, ValueError):
                print(f"  [llm] {model}: reply was empty or not valid JSON (try {attempt}), retrying...")
            except Exception as e:
                msg = str(e)
                if not _is_temporary(msg):
                    raise  # a real error (bad key, bad model name...) - stop and show it
                print(f"  [llm] {model}: busy / rate limited (try {attempt}). Reason: {msg[:150]}")
                if attempt < ATTEMPTS_PER_MODEL:
                    print(f"        waiting {wait}s...")
                    time.sleep(wait)
                    wait *= 2
        print(f"  [llm] giving up on {model}, trying the next model...")
    raise RuntimeError("All models failed. Wait a few minutes (free-tier limits reset) and run the cell again.")


In [ ]:
%%writefile check_setup.py
"""
check_setup.py - run this first to confirm everything works.
    python check_setup.py
"""
from src.llm import MODEL, ask_json, get_client, key_source

print("1) Checking API key...")
client = get_client()
print(f"   OK - key loaded from: {key_source()}")

print("\n2) Text-generation Flash models your key can use:")
for m in client.models.list():
    name = m.name.replace("models/", "")
    actions = getattr(m, "supported_actions", None) or []
    skip = any(w in name for w in ("tts", "image", "audio", "live", "embedding"))
    if "generateContent" in actions and "flash" in name and not skip:
        print("   -", name)

print(f"\n3) Test call to {MODEL} (cache off, so this is a real API call)...")
reply = ask_json('Return JSON exactly like this: {"status": "ok", "message": "<one short greeting>"}',
                 use_cache=False)
print("   Gemini replied:", reply)
print("\nAll good - you're ready to generate the test data.")


In [ ]:
%%writefile synthetic/scenario.py
"""
scenario.py - the GROUND TRUTH for our synthetic test.

Key idea: we write the answer key FIRST, then ask Gemini to write documents
that follow it. So we know exactly which requirement each slide covers,
which ones are only partly covered, and which are deliberately missing.

All companies here are fictional:
  - Client:     Qamar Logistics Group (Dubai)
  - Consultant: Meridian Advisory (fictional, so we never put made-up claims under a real firm's name)
"""

CLIENT = "Qamar Logistics Group"
CONSULTANT = "Meridian Advisory"
PROJECT = "AI Transformation Strategy & Roadmap"

# ---------------------------------------------------------------------------
# 1. What the RFP asks for (18 requirements across 4 categories)
# ---------------------------------------------------------------------------
REQUIREMENTS = [
    # Scope of work
    {"id": "S1", "category": "Scope", "text": "Assess the current AI and analytics maturity of all five business units (Ports, Freight, Warehousing, Customs Brokerage, Corporate Services)."},
    {"id": "S2", "category": "Scope", "text": "Identify at least 20 AI use cases and prioritise them by business value and feasibility, with an estimated financial impact for each."},
    {"id": "S3", "category": "Scope", "text": "Assess the readiness of the group's data infrastructure AND its data governance (data ownership, data quality standards and data policies)."},
    {"id": "S4", "category": "Scope", "text": "Benchmark the group's AI maturity against at least three regional and global logistics peers."},
    {"id": "S5", "category": "Scope", "text": "Design an AI target operating model, including an AI Centre of Excellence, key roles and decision rights."},
    {"id": "S6", "category": "Scope", "text": "Develop a Responsible AI framework aligned with the UAE's national AI ethics principles."},
    {"id": "S7", "category": "Scope", "text": "Design a change management and AI upskilling programme reaching approximately 500 employees."},
    {"id": "S8", "category": "Scope", "text": "Deliver two proof-of-concept pilots for top-priority use cases."},
    # Deliverables
    {"id": "D1", "category": "Deliverable", "text": "A current-state assessment report."},
    {"id": "D2", "category": "Deliverable", "text": "A prioritised AI use-case portfolio."},
    {"id": "D3", "category": "Deliverable", "text": "A three-year AI roadmap including investment estimates by year."},
    {"id": "D4", "category": "Deliverable", "text": "A final presentation to the Executive Board."},
    {"id": "D5", "category": "Deliverable", "text": "All final deliverables provided in both Arabic and English."},
    # Timeline
    {"id": "T1", "category": "Timeline", "text": "Complete the full engagement within 16 weeks."},
    {"id": "T2", "category": "Timeline", "text": "Hold an interim Steering Committee review at week 8."},
    {"id": "T3", "category": "Timeline", "text": "Mobilise the team and hold the project kick-off within two weeks of contract signature."},
    # Proposal requirements
    {"id": "C1", "category": "Proposal requirement", "text": "Provide CVs of the proposed team, including at least two Arabic-speaking consultants."},
    {"id": "C2", "category": "Proposal requirement", "text": "Provide fixed-fee pricing broken down by project phase."},
]

# ---------------------------------------------------------------------------
# 2. The deck plan: what each slide must (and must NOT) say
#    covers: {"S1": "full"} or {"S3": "partial"}
#    avoid:  words Gemini must not use on that slide (protects partials/gaps,
#            and forces paraphrasing so keyword matching struggles)
#    trap:   slides that SHARE KEYWORDS with a requirement but don't address it
# ---------------------------------------------------------------------------
SLIDES = [
    {"n": 1, "title": "Title slide", "covers": {},
     "brief": f"Title slide: '{PROJECT} - Proposal to {CLIENT}', submitted by {CONSULTANT}, October 2026. Subtitle only, max 2 short lines."},

    {"n": 2, "title": "About Meridian Advisory", "covers": {}, "trap": True,
     "brief": "Short firm profile and our values. Mention that our values are integrity, ethics and responsible conduct in how WE behave as a firm. Do NOT describe any AI ethics framework or any work for the client."},

    {"n": 3, "title": "Our understanding of your context", "covers": {},
     "brief": "Describe the client's business situation: regional growth, competition from digital-first logistics players, fragmented legacy systems. Describe the context only, NOT the work we will do."},

    {"n": 4, "title": "Our approach at a glance", "covers": {},
     "brief": "Four phases named Diagnose, Discover, Design, Deliver, with one short generic line each. No numbers, no weeks, no deliverable names."},

    {"n": 5, "title": "Phase 1 - Diagnose", "covers": {"S1": "full", "S3": "partial"},
     "brief": "Establish an AI readiness baseline across the five divisions (Ports, Freight, Warehousing, Customs Brokerage, Corporate Services) via interviews and a capability scorecard. Also review the TECHNICAL data platforms: data architecture, cloud infrastructure, system integration. Only the technical side of data.",
     "avoid": ["governance", "ownership", "data quality", "policy", "policies", "maturity"]},

    {"n": 6, "title": "Phase 2 - Discover", "covers": {"S2": "full"},
     "brief": "Build a long-list of 25+ AI opportunities across the business, score each on value vs. ease of implementation, and size each with an indicative EBITDA uplift. Call them 'opportunities' or 'AI plays'.",
     "avoid": ["use case", "use-case"]},

    {"n": 7, "title": "Phase 3 - Design", "covers": {"S5": "full", "D3": "partial"},
     "brief": "Design how AI will be run: an AI hub (centre of excellence), new roles (e.g. AI product owner, ML engineer), and who decides what. Also produce a 3-year sequenced roadmap in waves. The roadmap must NOT mention money in any way.",
     "avoid": ["investment", "budget", "cost", "capex", "AED", "funding", "operating model"]},

    {"n": 8, "title": "Phase 4 - Deliver", "covers": {"S8": "full"},
     "brief": "Build two rapid prototypes (minimum viable products) for the highest-ranked opportunities, tested with real users, with a go/no-go recommendation for scaling.",
     "avoid": ["proof of concept", "proof-of-concept", "pilot", "PoC"]},

    {"n": 9, "title": "Building AI capabilities", "covers": {"S7": "partial"},
     "brief": "Run AI awareness workshops for the senior leadership team only. Do NOT mention change management, staff numbers, or training for the wider workforce.",
     "avoid": ["500", "change management", "all employees", "workforce", "upskilling"]},

    {"n": 10, "title": "What you will receive", "covers": {"D1": "full", "D2": "full", "D4": "full"},
     "brief": "List of deliverables: a baseline diagnostic report on today's position, a ranked opportunity portfolio, and a final session with the Executive Board. Do NOT mention languages.",
     "avoid": ["Arabic", "bilingual", "translation"]},

    {"n": 11, "title": "Workplan", "covers": {"T1": "full", "T2": "full"}, "table": True,
     "brief": "Return a TABLE with columns Weeks | Phase | Key activities. Rows: weeks 1-4 Diagnose, weeks 5-8 Discover, weeks 9-13 Design, weeks 14-16 Deliver. State the total is 16 weeks. Include a mid-point checkpoint with the leadership sponsors in week 8. Do NOT mention kick-off or mobilisation.",
     "avoid": ["kick-off", "kickoff", "mobilis", "mobiliz", "Steering Committee", "contract"]},

    {"n": 12, "title": "Proposed team", "covers": {"C1": "partial"},
     "brief": "Five team members with role and a one-line bio each (fictional names). Note that full CVs are in the appendix. Do NOT mention languages spoken.",
     "avoid": ["Arabic", "language", "bilingual"]},

    {"n": 13, "title": "Commercial proposal", "covers": {"C2": "full"}, "trap": True,
     "brief": "Fixed fee per phase in AED (Diagnose, Discover, Design, Deliver) and the total. Payment milestones: 30% upon contract signature, the rest per phase. Say our fees are benchmarked against market rates.",
     "avoid": ["kick-off", "kickoff", "mobilis", "mobiliz"]},

    {"n": 14, "title": "Relevant experience", "covers": {}, "trap": True,
     "brief": "Three short case studies for OTHER fictional clients: an AI roadmap for a GCC bank, use cases identified for a Saudi retailer, and an Arabic-language chatbot for a government entity. Make clear these were for other clients."},

    {"n": 15, "title": "Why Meridian and next steps", "covers": {},
     "brief": "Three reasons to choose us and a closing line inviting a follow-up meeting. Generic, no new commitments."},
]


def expected_answer_key():
    """Turn the slide plan into the answer key: status + slide numbers per requirement."""
    key = []
    for req in REQUIREMENTS:
        full = [s["n"] for s in SLIDES if s["covers"].get(req["id"]) == "full"]
        partial = [s["n"] for s in SLIDES if s["covers"].get(req["id"]) == "partial"]
        if full:
            status = "Covered"
        elif partial:
            status = "Partially covered"
        else:
            status = "Not covered"
        key.append({**req, "expected_status": status, "expected_slides": full + partial})
    return key


In [ ]:
%%writefile synthetic/generate_test_data.py
"""
generate_test_data.py - creates the synthetic test set.

Run from the project root:
    python -m synthetic.generate_test_data

Output folders (kept separate on purpose):
  test_data/inputs/        -> the ONLY files the tool is allowed to read
  test_data/ground_truth/  -> answer key + raw Gemini output, used ONLY by
                              the Day 5 evaluation, never by the tool itself

Steps:
  1. Gemini writes the RFP text (as JSON)    -> we build .docx AND .pdf
  2. Gemini writes the slide text (as JSON)  -> we build .pptx
  3. We check Gemini followed the plan (no forbidden words on slides)
  4. We save the answer key (.json + .csv)

Gemini writes the WORDS; Python builds the FILES. That way the file structure
is always valid, and the answer key never depends on what Gemini decided.
"""
import csv
import json
from pathlib import Path
from xml.sax.saxutils import escape

from docx import Document
from pptx import Presentation
from pptx.util import Inches, Pt
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import ListFlowable, ListItem, Paragraph, SimpleDocTemplate, Spacer

from src.llm import ask_json
from synthetic.scenario import CLIENT, CONSULTANT, PROJECT, REQUIREMENTS, SLIDES, expected_answer_key

INPUTS = Path("test_data/inputs")
TRUTH = Path("test_data/ground_truth")

# ---------------------------------------------------------------------------
# Step 1: RFP
# ---------------------------------------------------------------------------
RFP_PROMPT = f"""
You are a procurement officer at {CLIENT}, a fictional logistics group in Dubai.
Write a realistic Request for Proposal (RFP) for "{PROJECT}".

It MUST contain every requirement below. Rules:
- Rephrase each in formal procurement language ("The Consultant shall...").
- Do NOT show the IDs (S1, D2...) in the document.
- Put about half of them as bullet points and weave the rest into paragraphs,
  so they are not all neatly listed.
- Also include normal RFP sections that are NOT consultant requirements:
  background, objectives, evaluation criteria with weights, submission
  instructions (deadline 20 October 2026), confidentiality.
- Do not add extra scope requirements beyond the list.
- Keep the whole RFP to roughly 1,500-2,000 words.

Requirements:
{json.dumps([{"id": r["id"], "text": r["text"]} for r in REQUIREMENTS], indent=1)}

Return JSON:
{{
  "title": "...",
  "reference": "RFP reference number",
  "sections": [{{"heading": "1. ...", "paragraphs": ["..."], "bullets": ["..."]}}],
  "requirement_locations": {{"S1": "heading of the section where it appears", "...": "..."}}
}}
"""


def generate_rfp():
    print("Step 1/4: asking Gemini to write the RFP...")
    rfp = ask_json(RFP_PROMPT, temperature=0.7, cache_tag="rfp")
    missing = [r["id"] for r in REQUIREMENTS if r["id"] not in rfp.get("requirement_locations", {})]
    if missing:
        print(f"  WARNING: Gemini did not confirm placing: {missing}. Check the RFP manually.")
    return rfp


def build_rfp_docx(rfp, path):
    doc = Document()
    doc.add_heading(str(rfp.get("title") or "Request for Proposal"), level=0)
    doc.add_paragraph(f"Reference: {rfp.get('reference') or ''}")
    for sec in rfp.get("sections") or []:
        doc.add_heading(str(sec.get("heading") or ""), level=1)
        for p in sec.get("paragraphs") or []:
            doc.add_paragraph(str(p))
        for b in sec.get("bullets") or []:
            doc.add_paragraph(str(b), style="List Bullet")
    doc.save(path)


def build_rfp_pdf(rfp, path):
    styles = getSampleStyleSheet()
    story = [Paragraph(escape(str(rfp.get("title") or "Request for Proposal")), styles["Title"]),
             Paragraph(escape(f"Reference: {rfp.get('reference') or ''}"), styles["Normal"]),
             Spacer(1, 12)]
    for sec in rfp.get("sections") or []:
        story.append(Paragraph(escape(str(sec.get("heading") or "")), styles["Heading2"]))
        for p in sec.get("paragraphs") or []:
            story.append(Paragraph(escape(str(p)), styles["BodyText"]))
            story.append(Spacer(1, 6))
        bullets = sec.get("bullets") or []
        if bullets:
            story.append(ListFlowable(
                [ListItem(Paragraph(escape(str(b)), styles["BodyText"])) for b in bullets],
                bulletType="bullet"))
    SimpleDocTemplate(str(path), pagesize=A4).build(story)


# ---------------------------------------------------------------------------
# Step 2: Deck
# ---------------------------------------------------------------------------
def deck_prompt():
    plan = [{"slide_number": s["n"], "working_title": s["title"], "brief": s["brief"],
             "forbidden_words": s.get("avoid", []), "needs_table": s.get("table", False)}
            for s in SLIDES]
    return f"""
You are a consultant at {CONSULTANT} (a fictional firm) writing a proposal deck
for {CLIENT} on "{PROJECT}".

Write the text for EXACTLY {len(SLIDES)} slides, following each brief closely.
Rules:
- Use confident consulting language and your OWN vocabulary.
- Never use a slide's forbidden_words (or close variants) on that slide.
- Never cover something a brief does not ask for.
- 3-5 bullets per slide, each under 25 words. Short speaker notes per slide.
- If needs_table is true, return the content in "table" and keep bullets short.

Slide plan:
{json.dumps(plan, indent=1)}

Return JSON:
{{"slides": [{{"slide_number": 1, "title": "...", "bullets": ["..."], "notes": "...",
              "table": {{"headers": ["..."], "rows": [["..."]]}} or null}}]}}
"""


def check_deck(slides):
    """Find forbidden words Gemini used anyway. Returns a list of problems."""
    problems = []
    if len(slides) != len(SLIDES):
        problems.append(f"Expected {len(SLIDES)} slides, got {len(slides)}")
    for plan, slide in zip(SLIDES, slides):
        text = json.dumps(slide).lower()
        for word in plan.get("avoid", []):
            if word.lower() in text:
                problems.append(f"Slide {plan['n']} uses forbidden word '{word}'")
    return problems


def generate_deck(max_tries=3):
    print("Step 2/4: asking Gemini to write the deck...")
    best, best_problems = None, None
    for attempt in range(1, max_tries + 1):
        # A different cache_tag per attempt, so a retry asks Gemini for a NEW deck
        # instead of getting the same cached (rule-breaking) one back.
        reply = ask_json(deck_prompt(), temperature=0.7, cache_tag=f"deck-attempt-{attempt}")
        slides = reply.get("slides") if isinstance(reply, dict) else reply
        slides = slides if isinstance(slides, list) else []
        problems = check_deck(slides)
        print(f"  attempt {attempt}: {len(problems)} rule violation(s)")
        if best is None or len(problems) < len(best_problems):
            best, best_problems = slides, problems
        if not problems:
            break
    return best, best_problems


def build_deck(slides, path):
    """
    Turn Gemini's slide text into a real .pptx. Slides are placed in the order
    given (position = slide number), and missing/empty fields are tolerated.
    """
    prs = Presentation()
    prs.slide_width, prs.slide_height = Inches(13.333), Inches(7.5)
    for i, s in enumerate(slides, start=1):
        title = str(s.get("title") or f"Slide {i}")
        bullets = [str(b) for b in (s.get("bullets") or [])]
        table = s.get("table") or {}
        headers = [str(h) for h in (table.get("headers") or [])]
        rows = [r for r in (table.get("rows") or []) if isinstance(r, list)]

        if i == 1:
            slide = prs.slides.add_slide(prs.slide_layouts[0])  # title layout
            slide.shapes.title.text = title
            slide.placeholders[1].text = "\n".join(bullets)
        elif headers and rows:
            slide = prs.slides.add_slide(prs.slide_layouts[5])  # title only + table
            slide.shapes.title.text = title
            shape = slide.shapes.add_table(len(rows) + 1, len(headers),
                                           Inches(0.6), Inches(1.6), Inches(12), Inches(4))
            for c, h in enumerate(headers):
                shape.table.cell(0, c).text = h
            for r, row in enumerate(rows, start=1):
                for c, val in enumerate(row[:len(headers)]):
                    shape.table.cell(r, c).text = str(val)
            if bullets:  # short caption under the table
                box = slide.shapes.add_textbox(Inches(0.6), Inches(5.9), Inches(12), Inches(1))
                box.text_frame.text = " | ".join(bullets)
        else:
            slide = prs.slides.add_slide(prs.slide_layouts[1])  # title + bullets
            slide.shapes.title.text = title
            body = slide.placeholders[1].text_frame
            body.text = bullets[0] if bullets else ""
            for b in bullets[1:]:
                body.add_paragraph().text = b
            for p in body.paragraphs:
                for run in p.runs:
                    run.font.size = Pt(18)
        slide.notes_slide.notes_text_frame.text = str(s.get("notes") or "")
    prs.save(path)


# ---------------------------------------------------------------------------
# Step 3 + 4: answer key
# ---------------------------------------------------------------------------
def save_answer_key(rfp):
    key = expected_answer_key()
    locations = rfp.get("requirement_locations", {})
    for row in key:
        row["rfp_section"] = locations.get(row["id"], "?")
    (TRUTH / "answer_key.json").write_text(json.dumps(key, indent=2), encoding="utf-8")
    with open(TRUTH / "answer_key.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["id", "category", "text", "expected_status",
                                          "expected_slides", "rfp_section"])
        w.writeheader()
        for row in key:
            w.writerow({**row, "expected_slides": ", ".join(map(str, row["expected_slides"]))})
    return key


def main():
    INPUTS.mkdir(parents=True, exist_ok=True)
    TRUTH.mkdir(parents=True, exist_ok=True)

    rfp = generate_rfp()
    (TRUTH / "rfp_content.json").write_text(json.dumps(rfp, indent=2), encoding="utf-8")
    build_rfp_docx(rfp, INPUTS / "rfp_qamar.docx")
    build_rfp_pdf(rfp, INPUTS / "rfp_qamar.pdf")

    slides, problems = generate_deck()
    (TRUTH / "deck_content.json").write_text(json.dumps(slides, indent=2), encoding="utf-8")
    build_deck(slides, INPUTS / "proposal_qamar.pptx")

    print("Step 3/4: checking the deck follows the plan...")
    for p in problems:
        print("  WARNING:", p)
    if not problems:
        print("  OK - no rule violations")

    print("Step 4/4: saving the answer key...")
    key = save_answer_key(rfp)
    counts = {}
    for row in key:
        counts[row["expected_status"]] = counts.get(row["expected_status"], 0) + 1
    print(f"\nDone! Tool inputs in '{INPUTS}/', answer key in '{TRUTH}/'.")
    print(f"Planted answer key: {counts}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile synthetic/verify_test_data.py
"""
verify_test_data.py - checks the generated test set is complete and correct.

Run from the project root:
    python -m synthetic.verify_test_data

It opens the REAL files (not Gemini's raw JSON), so it checks what the tool
will actually see. Every check prints a tick or a cross.
"""
import json
from pathlib import Path

import pdfplumber
from docx import Document
from pptx import Presentation

from synthetic.scenario import REQUIREMENTS, SLIDES

INPUTS = Path("test_data/inputs")
TRUTH = Path("test_data/ground_truth")
results = []


def check(ok, label):
    results.append(ok)
    print(("  ✅ " if ok else "  ❌ ") + label)
    return ok


def slide_text(slide):
    """All text on a slide: text boxes, tables and speaker notes."""
    parts = []
    for shape in slide.shapes:
        if shape.has_text_frame:
            parts.append(shape.text_frame.text)
        if shape.has_table:
            for row in shape.table.rows:
                parts.extend(cell.text for cell in row.cells)
    if slide.has_notes_slide:
        parts.append(slide.notes_slide.notes_text_frame.text)
    return " ".join(parts)


print("1) All files exist")
files = [INPUTS / "rfp_qamar.docx", INPUTS / "rfp_qamar.pdf", INPUTS / "proposal_qamar.pptx",
         TRUTH / "answer_key.json", TRUTH / "answer_key.csv",
         TRUTH / "rfp_content.json", TRUTH / "deck_content.json"]
for f in files:
    check(f.exists() and f.stat().st_size > 0, str(f))
if not all(results):
    raise SystemExit("\nFiles are missing - run the 'Generate test data' cell first.")

print("\n2) RFP is readable and complete")
pdf_text = "\n".join(page.extract_text() or "" for page in pdfplumber.open(INPUTS / "rfp_qamar.pdf").pages)
docx_words = sum(len(p.text.split()) for p in Document(INPUTS / "rfp_qamar.docx").paragraphs)
check(len(pdf_text.split()) > 500, f"PDF text readable ({len(pdf_text.split())} words)")
check(docx_words > 500, f"Word text readable ({docx_words} words)")
locations = json.loads((TRUTH / "rfp_content.json").read_text(encoding="utf-8")).get("requirement_locations", {})
missing = [r["id"] for r in REQUIREMENTS if r["id"] not in locations]
check(not missing, "Gemini placed all 18 requirements in the RFP" + (f" (missing: {missing})" if missing else ""))

print("\n3) Deck structure")
prs = Presentation(INPUTS / "proposal_qamar.pptx")
slides = list(prs.slides)
check(len(slides) == len(SLIDES), f"{len(slides)} slides (expected {len(SLIDES)})")
check(any(sh.has_table for sh in slides[10].shapes) if len(slides) > 10 else False, "Slide 11 contains a table")
with_notes = sum(1 for s in slides if s.has_notes_slide and s.notes_slide.notes_text_frame.text.strip())
print(f"  ℹ️ {with_notes}/{len(slides)} slides have speaker notes (information only)")

print("\n4) Planted gaps are intact (no forbidden words in the real slides)")
for plan, slide in zip(SLIDES, slides):
    if plan.get("avoid"):
        text = slide_text(slide).lower()
        used = [w for w in plan["avoid"] if w.lower() in text]
        check(not used, f"Slide {plan['n']} ({plan['title']})" + (f" uses: {used}" if used else ""))

print("\n5) Answer key")
key = json.loads((TRUTH / "answer_key.json").read_text(encoding="utf-8"))
counts = {s: sum(r["expected_status"] == s for r in key) for s in ("Covered", "Partially covered", "Not covered")}
check(len(key) == 18, f"{len(key)} requirements in the answer key")
check(counts == {"Covered": 10, "Partially covered": 4, "Not covered": 4}, f"Status split: {counts}")

failed = results.count(False)
print("\n" + ("🎉 ALL CHECKS PASSED - the test data is complete." if failed == 0
              else f"⚠️ {failed} check(s) failed - paste this output to Claude."))


In [ ]:
%%writefile DESIGN_DECISIONS.md
# Design decisions log

A running record of what I decided, why, and what I considered instead.

## Day 1: Setup and synthetic test data

**1. Answer key written before the documents.**
The requirements and the slide-by-slide coverage plan are defined in code (`synthetic/scenario.py`), and Gemini writes documents that follow the plan. *Why:* if the AI generated the RFP and deck freely, I wouldn't know the correct answers, so I couldn't measure accuracy. *Alternative:* generate freely and label by hand afterwards, which is slower and less reliable.

**2. The test set is designed to break keyword matching.**
It has 10 covered, 4 partial and 4 missing requirements, plus paraphrases (with forbidden words, so slides can't reuse the RFP's vocabulary) and 3 trap slides that share keywords without addressing the requirement. *Why:* these target the two failure modes named in the brief: missed matches when wording differs, and false matches on shared words.

**3. Gemini writes the words, Python builds the files.**
Gemini returns JSON content, and python-docx, reportlab and python-pptx turn it into the files. *Why:* the files are always valid, and the structure (number of slides, tables, notes) is controlled. The RFP is produced as both Word and PDF so both input types can be tested.

**4. Generated content is checked automatically.**
A forbidden-word check runs on every slide, and the deck is regenerated up to 3 times if it breaks the rules. *Why:* the AI output is validated rather than trusted blindly. This is followed by a manual review of the partial slides.

**5. Tool inputs and ground truth are kept in separate folders.**
`test_data/inputs/` holds what the tool reads; `test_data/ground_truth/` holds the answer key, used only by the evaluation. *Why:* this prevents data leakage. The tool must never see the answers it is graded against.

**6. All LLM calls go through one module (`src/llm.py`).**
*Why:* the key handling, retries, caching and model choice live in one place. Switching to another provider (e.g. an enterprise or on-premise model for confidential client data) means changing a single file.

**7. The API key is never in the code.**
It is read from a `.env` file locally (and `.env` is in `.gitignore`) or from Kaggle Secrets. The setup check reports which source was used.

**8. Responses are cached to disk.**
Each Gemini answer is saved under a fingerprint of model + prompt + settings. *Why:* reruns cost nothing on the free tier, and the demo gives identical results every time (reproducibility). Changing the prompt or model automatically creates a new cache entry.

**9. JSON mode plus exponential backoff.**
Gemini is forced to answer in JSON, and on rate-limit errors the code waits 10s, then 20s, then 40s, and so on. *Why:* the free tier allows only about 10 requests per minute, so the tool slows down instead of crashing.

**10. Model: gemini-3.8-flash, with an automatic backup (gemini-3.5-flash-lite).**
Both are pinned versions, not "-latest" aliases. *Why:* a pinned version keeps results reproducible, whereas an alias can silently change which model is used. *Lesson learned:* the newest model was sometimes too busy for large requests, so the code now retries 3 times, prints the real error reason, and then switches to the backup model by itself. The cache records which model actually answered.

**11. The whole project lives in one notebook.**
Every source file is written by a `%%writefile` cell. *Why:* uploading the code as a Kaggle dataset caused nested-folder problems. One self-contained notebook is simpler to run, share and reproduce, and the files it writes are the same ones that go to GitHub.

**12. An automatic checklist verifies the test data.**
`synthetic/verify_test_data.py` opens the real Word, PDF and PPTX files (not Gemini's raw JSON) and checks that every file exists, the text is readable, there are 15 slides with a table on slide 11 and notes on every slide, no forbidden words appear, and the answer key is 10/4/4. *Why:* it confirms the test set is complete in seconds, instead of relying on manual checking.

## Risks and limitations noted so far
- Synthetic data is cleaner than real client documents (real RFPs have scanned pages, annexes, inconsistent formatting).
- One scenario means one test set. A second scenario would make the results stronger, if time allows.
- The free tier may use prompts to improve Google's products, which is acceptable for synthetic data but not for real client RFPs.


## 3. Install libraries
Warnings about `dopamine-rl` or `gradio` are harmless.

In [ ]:
!pip install -q -r requirements.txt

## 4. Load the API key and choose models

In [ ]:
from kaggle_secrets import UserSecretsClient

# Read the key from Kaggle Secrets and hand it to our scripts as an environment variable
try:
    os.environ['GEMINI_API_KEY'] = UserSecretsClient().get_secret('GEMINI_API_KEY')
    print('✅ Key loaded from Kaggle Secrets (length:', len(os.environ['GEMINI_API_KEY']), 'characters)')
except Exception as e:
    print('❌ Could not read the secret. Check Add-ons > Secrets: GEMINI_API_KEY must be TICKED.')
    raise

os.environ['GEMINI_MODEL'] = 'gemini-3.8-flash'                 # main model
os.environ['GEMINI_FALLBACK_MODELS'] = 'gemini-3.5-flash-lite'  # used automatically if the main one is busy

## 5. Check the connection

In [ ]:
!python check_setup.py

## 6. Generate the test data
Takes a few minutes. 'Busy / rate limited' messages are normal: the code waits, and switches to the backup model if needed.

In [ ]:
!python -m synthetic.generate_test_data

## 7. Checklist: is everything there and correct?

In [ ]:
!python -m synthetic.verify_test_data

## 8. Review the answer key and the partial slides

In [ ]:
import pandas as pd
from pptx import Presentation
from synthetic.scenario import SLIDES

# 1) The answer key (what the tool SHOULD find)
display(pd.read_csv('test_data/ground_truth/answer_key.csv'))

# 2) The 4 partial slides, read from the REAL .pptx, with what each was told to leave out
deck = list(Presentation('test_data/inputs/proposal_qamar.pptx').slides)
for plan in SLIDES:
    if 'partial' in plan['covers'].values():
        n = plan['n']
        print(f"\n=== Slide {n}: {plan['title']} | must NOT mention: {', '.join(plan.get('avoid', []))}")
        if n > len(deck):
            print('  (slide missing from deck)')
            continue
        for shape in deck[n - 1].shapes:
            if shape.has_text_frame and shape.text_frame.text.strip():
                for line in shape.text_frame.text.splitlines():
                    print('  -', line)

## 9. Download everything

In [ ]:
import shutil

# Zip the whole project: code + test data + cache (the cache makes reruns free and identical)
shutil.make_archive('/kaggle/working/rfp_coverage_mapper_day1', 'zip', '/kaggle/working', 'rfp-coverage-mapper')
print('✅ Saved rfp_coverage_mapper_day1.zip - download it from the Output panel, then click Save Version.')